# Demo 3: a quantum kernel SVM on two moons
Slide 25 (Demo 3 of 3), after slides 23–24 (Quantum ML). Same SVM, two kernels: classical RBF and a quantum feature map.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from demo_utils import GREEN, LIGHT

X, y = make_moons(200, noise=0.15, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=100, random_state=0, stratify=y)

scaler = MinMaxScaler((0, np.pi)).fit(X_train)     # features become rotation angles
A_train, A_test = scaler.transform(X_train), scaler.transform(X_test)

fig, ax = plt.subplots(figsize=(5, 3.4))
ax.scatter(*X_train[y_train == 0].T, color=LIGHT, edgecolor="white", s=40)
ax.scatter(*X_train[y_train == 1].T, color=GREEN, edgecolor="white", s=40)
ax.set_title("Two moons: 100 training points")
plt.show()

### Baseline: classical SVM with an RBF kernel

In [ ]:
t = time.perf_counter()
rbf = SVC(kernel="rbf").fit(X_train, y_train)
print(f"RBF accuracy: {rbf.score(X_test, y_test):.2f}   time: {time.perf_counter() - t:.3f} s")

### Quantum kernel: k(x, x′) = |⟨φ(x′)|φ(x)⟩|²
Encode x, un-encode x′, measure the probability of getting back to |00⟩.

In [ ]:
dev = qml.device("default.qubit", wires=2)

@qml.qnode(dev)
def kernel_circuit(x1, x2):
    qml.IQPEmbedding(x1, wires=range(2), n_repeats=2)               # feature map φ(x1)
    qml.adjoint(qml.IQPEmbedding)(x2, wires=range(2), n_repeats=2)  # undo φ(x2)
    return qml.probs(wires=range(2))

def quantum_kernel(x1, x2):
    return kernel_circuit(x1, x2)[0]

print(qml.draw(kernel_circuit, level="top", decimals=None)(A_train[0], A_train[1]))

### Train: one circuit per pair of training points

In [ ]:
t = time.perf_counter()
K_train = qml.kernels.square_kernel_matrix(A_train, quantum_kernel)
n = len(A_train)
print(f"{n * (n - 1) // 2:,} circuits   time: {time.perf_counter() - t:.1f} s")

qsvm = SVC(kernel="precomputed").fit(K_train, y_train)

In [ ]:
def kernel_matrix(A, B):
    # same circuits, batched: one call per training point
    return np.array([kernel_circuit(A, b)[:, 0] for b in B]).T

K_test = kernel_matrix(A_test, A_train)
print(f"Quantum kernel accuracy: {qsvm.score(K_test, y_test):.2f}")
print(f"RBF accuracy:            {rbf.score(X_test, y_test):.2f}")

### What each model learned

In [ ]:
xx, yy = np.meshgrid(np.linspace(-1.6, 2.6, 50), np.linspace(-1.1, 1.6, 50))
grid = np.c_[xx.ravel(), yy.ravel()]
z_rbf = rbf.decision_function(grid).reshape(xx.shape)
z_q = qsvm.decision_function(kernel_matrix(scaler.transform(grid), A_train)).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, z, name, acc in [(axes[0], z_rbf, "Classical RBF", rbf.score(X_test, y_test)),
                         (axes[1], z_q, "Quantum kernel", qsvm.score(K_test, y_test))]:
    ax.contourf(xx, yy, z > 0, levels=1, colors=["#F2F2F2", "#EAF7EA"])
    ax.contour(xx, yy, z, levels=[0], colors=[GREEN], linewidths=2)
    ax.scatter(*X_test[y_test == 0].T, color=LIGHT, edgecolor="white", s=35)
    ax.scatter(*X_test[y_test == 1].T, color=GREEN, edgecolor="white", s=35)
    ax.set_title(f"{name}: test accuracy {acc:.2f}")
plt.show()

> Same data, same SVM. The quantum kernel is not better here, and it needed thousands of circuits.
> On a real QPU each of them also costs ~1,000 shots: the O(N²) from slide 24 in practice.